# Bengaluru House Price Prediction

Data: `BHP.csv`, supplied as `BHP(2).xls` in CSV format. Price is assumed to be in INR lakh, following the original notebook. The collection date and original source documentation are unavailable.

Inputs: location, area type, floor area, bedrooms and bathrooms. Model selection uses five-fold grouped cross-validation on training data; 20% of input-description groups are held out for testing. The selection metric is MAE.

## Imports

### Basic libraries

In [ ]:
import numpy as np
import pandas as pd


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [3]:
SEED = 42

## Load data

In [4]:
raw = pd.read_csv('BHP.csv')

## Inspect the data

In [5]:
print(raw.head().to_string(index=False))

           area_type  availability                 location      size society total_sqft  bath  balcony  price
Super built-up  Area        19-Dec Electronic City Phase II     2 BHK Coomee        1056   2.0      1.0  39.07
          Plot  Area Ready To Move         Chikka Tirupathi 4 Bedroom Theanmp       2600   5.0      3.0 120.00
      Built-up  Area Ready To Move              Uttarahalli     3 BHK     NaN       1440   2.0      3.0  62.00
Super built-up  Area Ready To Move       Lingadheeranahalli     3 BHK Soiewre       1521   3.0      1.0  95.00
Super built-up  Area Ready To Move                 Kothanur     2 BHK     NaN       1200   2.0      1.0  51.00


In [6]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [7]:
print(raw.describe(include='all').to_string())

                   area_type   availability    location   size  society total_sqft          bath       balcony         price
count                  13320          13320       13319  13304     7818      13320  13247.000000  12711.000000  13320.000000
unique                     4             81        1305     31     2688       2117           NaN           NaN           NaN
top     Super built-up  Area  Ready To Move  Whitefield  2 BHK  GrrvaGr       1200           NaN           NaN           NaN
freq                    8790          10581         540   5199       80        843           NaN           NaN           NaN
mean                     NaN            NaN         NaN    NaN      NaN        NaN      2.692610      1.584376    112.565627
std                      NaN            NaN         NaN    NaN      NaN        NaN      1.341458      0.817263    148.971674
min                      NaN            NaN         NaN    NaN      NaN        NaN      1.000000      0.000000      8.000000


## Data overview

In [8]:
print('Raw rows and columns:', raw.shape)
print('Exact duplicate source rows:', raw.duplicated().sum())

Raw rows and columns: (13320, 9)
Exact duplicate source rows: 529


In [ ]:
print('Missing values:\n', raw.isna().sum().to_string())


## Area conversion

### Convert areas to square feet

In [10]:
def parse_area(value):
    text = str(value).strip()

    if '-' in text:
        parts = text.split('-')
        try:
            low = float(parts[0])
            high = float(parts[1])
            return (low + high) / 2
        except ValueError:
            return np.nan

    units = {
        'Sq. Meter': 10.7639104,
        'Sq. Yards': 9,
        'Acres': 43560,
        'Guntha': 1089,
        'Grounds': 2400,
        'Perch': 272.25,
        'Cents': 435.6
    }

    for unit in units:
        if text.endswith(unit):
            number = text.replace(unit, '').strip()
            try:
                return float(number) * units[unit]
            except ValueError:
                return np.nan

    try:
        return float(text)
    except ValueError:
        return np.nan

## Data cleaning

In [11]:
df = raw.drop_duplicates().copy()

In [12]:
df['total_sqft'] = df['total_sqft'].map(parse_area)

### Extract bedrooms: “2 BHK” becomes 2

In [13]:
bedrooms = df['size'].str.split().str[0]
df['bhk'] = pd.to_numeric(bedrooms, errors='coerce')

### Convert bathrooms and price to numbers

In [14]:
df['bath'] = pd.to_numeric(df['bath'], errors='coerce')
df['price'] = pd.to_numeric(df['price'], errors='coerce')

### Remove extra spaces from text

In [15]:
def clean_text(value):
    if isinstance(value, str):
        return value.strip()
    return np.nan

df['location'] = df['location'].apply(clean_text)
df['area_type'] = df['area_type'].apply(clean_text)
df['location'] = df['location'].replace('', np.nan)
df['area_type'] = df['area_type'].replace('', np.nan)

### Check valid prices, areas and bedroom counts

In [16]:
valid_price = df['price'].notna()
valid_price = valid_price & np.isfinite(df['price'])
valid_price = valid_price & (df['price'] > 0)

valid_area = df['total_sqft'].isna() | (df['total_sqft'] > 0)
valid_bhk = df['bhk'].isna() | (df['bhk'] > 0)

valid = valid_price & valid_area & valid_bhk
print('Rows excluded:', len(df) - valid.sum())

Rows excluded: 0


## Feature selection

### Keep the columns used in this model

In [17]:
features = ['location', 'area_type', 'total_sqft', 'bhk', 'bath']
columns = features + ['price']
df = df.loc[valid, columns]
df = df.drop_duplicates()
df = df.reset_index(drop=True)

### X contains inputs; y contains prices

In [18]:
X = df[features]
y = df['price']

## Duplicate groups

Each identical set of inputs gets the same group code. The hashing line is a pandas utility; its purpose is to keep repeated descriptions together.

In [ ]:
groups = pd.util.hash_pandas_object(X, index=False).to_numpy()
print('Final records:', len(df), '| distinct input descriptions:', len(set(groups)))


## Train–test split

### Reserve 20% of description groups for testing

In [21]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
for train_idx, test_idx in splitter.split(X, y, groups):
    break

### Create separate training and test variables

In [22]:
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]
train_groups = groups[train_idx]
test_groups = groups[test_idx]

In [ ]:

print('Train rows:', len(X_train), '| test rows:', len(X_test))

## Preprocessing

`Pipeline` runs steps in order. `ColumnTransformer` applies different steps to numbers and text. Keeping these inside the model search prevents validation data from affecting the learned cleaning rules.

### Fill missing numbers, then scale them

In [30]:
number_filler = SimpleImputer(strategy='median')
number_scaler = StandardScaler()

numeric_steps = [('fill', number_filler), ('scale', number_scaler)]
numeric = Pipeline(numeric_steps)

### Fill missing categories, then encode them

In [31]:
text_filler = SimpleImputer(strategy='most_frequent')
text_encoder = OneHotEncoder(
    handle_unknown='infrequent_if_exist',
    min_frequency=10
)

category_steps = [('fill', text_filler), ('encode', text_encoder)]
categorical = Pipeline(category_steps)

### Apply each preparation to its columns

In [32]:
numeric_columns = ['total_sqft', 'bhk', 'bath']
category_columns = ['location', 'area_type']

preparation_steps = [
    ('numbers', numeric, numeric_columns),
    ('categories', categorical, category_columns)
]
preprocess = ColumnTransformer(preparation_steps)

### Connect preparation and prediction

In [33]:
baseline_model = DummyRegressor(strategy='median')
model_steps = [('prepare', preprocess), ('model', baseline_model)]
pipeline = Pipeline(model_steps)

In [34]:
cv = GroupKFold(n_splits=5)

## Model comparison

### Create the candidate models

In [ ]:

linear = LinearRegression()
lasso = Lasso(max_iter=20000)
tree = DecisionTreeRegressor(random_state=SEED)
forest = RandomForestRegressor(n_estimators=80, random_state=SEED, n_jobs=1)

### Choose the settings to compare

In [ ]:

linear_settings = {'model': [linear]}
lasso_settings = {'model': [lasso], 'model__alpha': [0.1, 1.0]}

tree_settings = {
    'model': [tree],
    'model__max_depth': [8, 16],
    'model__min_samples_leaf': [5]
}

forest_settings = {
    'model': [forest],
    'model__max_depth': [16, None],
    'model__min_samples_leaf': [1, 3]
}

grid = [linear_settings, lasso_settings, tree_settings, forest_settings]

`GridSearchCV` tries the settings using training folds. `refit="mae"` selects the lowest average absolute error and refits that choice on training data. The held-out test set is not used here.

In [37]:
search = GridSearchCV(pipeline, grid, cv=cv,
                      scoring={'mae': 'neg_mean_absolute_error', 'r2': 'r2'},
                      refit='mae', n_jobs=1, error_score='raise')

In [38]:
search.fit(X_train, y_train, groups=train_groups)

GridSearchCV(cv=GroupKFold(n_splits=5, random_state=None, shuffle=False),
             error_score='raise',
             estimator=Pipeline(steps=[('prepare',
                                        ColumnTransformer(transformers=[('numbers',
                                                                         Pipeline(steps=[('fill',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scale',
                                                                                          StandardScaler())]),
                                                                         ['total_sqft',
                                                                          'bhk',
                                                                          'bath']),
                                                                        ('categories',
                                                                         Pipeline(steps=[('fill',
                                                                                          SimpleImputer(strategy='m...
                         {'model': [Lasso(max_iter=20000)],
                          'model__alpha': [0.1, 1.0]},
                         {'model': [DecisionTreeRegressor(random_state=42)],
                          'model__max_depth': [8, 16],
                          'model__min_samples_leaf': [5]},
                         {'model': [RandomForestRegressor(n_estimators=80,
                                                          n_jobs=1,
                                                          random_state=42)],
                          'model__max_depth': [16, None],
                          'model__min_samples_leaf': [1, 3]}],
             refit='mae',
             scoring={'mae': 'neg_mean_absolute_error', 'r2': 'r2'})

## Cross-validation results

### Make a model-comparison table

In [ ]:
cv_results = pd.DataFrame(search.cv_results_)
model_names = []
for model in cv_results['param_model']:
    model_names.append(model.__class__.__name__)

comparison = pd.DataFrame()
comparison['model'] = model_names
comparison['CV_MAE_lakh'] = -cv_results['mean_test_mae']
comparison['CV_R2'] = cv_results['mean_test_r2']

comparison = comparison.sort_values('CV_MAE_lakh')

In [ ]:
print(comparison.to_string(index=False))
print('Selected using training CV:', search.best_params_)

## Test evaluation

In [41]:
best_model = search.best_estimator_
prediction = best_model.predict(X_test)

### Measure the selected model

In [43]:
model_r2 = r2_score(y_test, prediction)



print('Model R2:', round(model_r2, 4))



Model R2: 0.6456


## Price prediction

### Predict one property

In [47]:
def predict_price(location, total_sqft, bhk, bath,
                  area_type='Super built-up  Area'):
    if total_sqft <= 0 or bhk <= 0 or bath <= 0:
        raise ValueError('Area, bedrooms and bathrooms must be positive.')

    property_details = {
        'location': [location.strip()],
        'area_type': [area_type.strip()],
        'total_sqft': [total_sqft],
        'bhk': [bhk],
        'bath': [bath]
    }
    new_property = pd.DataFrame(property_details)
    estimated_price = best_model.predict(new_property)
    return float(estimated_price[0])

In [48]:
example = predict_price('Whitefield', 1200, 2, 2)
print(f'Example: Whitefield, 1200 sqft, 2 BHK, 2 bathrooms: {example:.2f} lakh')

Example: Whitefield, 1200 sqft, 2 BHK, 2 bathrooms: 59.00 lakh
